# 0.0 Load Dataframes

In [ ]:
import sys
sys.path.insert(0, '/home/ethantu/workspace/good-vibrations/src3')

import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from post_process import normalize_fft

In [ ]:
def _sample_normalize_mode(sample_dir, atol=0.05):
    # Check this sample's metadata.jsonl for a recorded normalize_mode; else infer from
    # its own X.npy stats (std-sample: std~=1; z-sample: std~=1 and mean~=0; see
    # post_process.normalize_fft). Returns None if X.npy is missing or inconclusive.
    meta_path = Path(sample_dir) / 'metadata.jsonl'
    if meta_path.exists():
        for line in meta_path.read_text().strip().splitlines():
            if 'normalize_mode' in (meta := json.loads(line)):
                return meta['normalize_mode']

    x_path = Path(sample_dir) / 'X.npy'
    if not x_path.exists():
        return None
    x = np.load(x_path)
    std, mean = x.std(), x.mean()
    if not np.isclose(std, 1.0, atol=atol):
        return None
    return 'z-sample' if np.isclose(mean, 0.0, atol=atol) else 'std-sample'

def get_normalize_mode(sample_dirs):
    # Compute normalize_mode per sample and assert the whole dataset agrees, so we get
    # one dataset-wide value rather than trusting/checking each sample individually.
    modes = {_sample_normalize_mode(d) for d in sample_dirs}
    assert len(modes) == 1, f'inconsistent normalize_mode across samples: {modes}'
    return modes.pop()

In [ ]:
BASE_SAMPLE_DIR = '/home/ethantu/workspace/good-vibrations/data/samples'

# Optional: a run's forward_outputs/ dir (dir containing 'train' and 'eval/*' subdirs of
# batched .pt files saved during eval). Set to None to skip loading run predictions.
RUN_DIR = '/home/ethantu/workspace/good-vibrations/runs/fix-dropout/outputs'

In [ ]:
# One row per sample dir under BASE_SAMPLE_DIR, built from each sample's metadata.jsonl.
# fft_path points at the raw (unprocessed, complex) per-laser FFT shifts -- shape
# (1, n_lasers, n_freqs, 2), last dim = (x, y) shift direction -- loaded lazily by
# compute_fft_magnitude rather than eagerly here, since each file is a few MB.
# com is the downsampled (mask-space) center of mass, matching the rest of the notebook.
# out_h/out_w are the downsampled mask grid dimensions com_x/com_y live in (see
# post_process.downsample) -- used as fixed COM plot axis ranges so plots don't
# auto-zoom to whatever subset of points is currently filtered.
# normalize_mode is computed once for the whole dataset (get_normalize_mode asserts every
# sample agrees) and assigned to every row.

def load_samples_df(base_sample_dir=BASE_SAMPLE_DIR):
    sample_dirs = [d for d in sorted(Path(base_sample_dir).iterdir())
                  if (d / 'metadata.jsonl').exists() and (d / 'inputs' / '03_fft_shifts.npz').exists()]
    normalize_mode = get_normalize_mode(sample_dirs)

    rows = []
    for sample_dir in sample_dirs:
        meta = {}
        for line in (sample_dir / 'metadata.jsonl').read_text().strip().splitlines():
            meta |= json.loads(line)
        com = meta.get('downsampled_com')
        rows.append({
            'sample_id':      int(meta['sample_id']),
            'output_id':      meta.get('output_id'),
            'speaker':        meta.get('speaker'),
            'n_objects':      meta.get('n_objects'),
            'box':            meta.get('box'),
            'object':         meta.get('object'),
            'is_empty_box':   meta.get('is_empty_box'),
            'com_x':          com[0] if com is not None else None,
            'com_y':          com[1] if com is not None else None,
            'out_h':          meta.get('out_h'),
            'out_w':          meta.get('out_w'),
            'fft_path':       sample_dir / 'inputs' / '03_fft_shifts.npz',
            'overhead_path':  sample_dir / 'overhead.png',
            'normalize_mode': normalize_mode,
        })
    return pd.DataFrame(rows)

samples_df = load_samples_df()
samples_df

In [ ]:
# Load a run's forward_outputs (RUN_DIR): walk 'train' and every 'eval/*' subdir,
# read each batched .pt file (keys: fft, mask_pred, mask_true, info), and flatten to
# one row per sample with its split, predicted center of mass, distance from the true
# com, and mask MSE. Returns None if RUN_DIR is None (no run to compare against).

def _mask_com(mask):
    # (row, col) center of mass of a single 2D mask, same convention as
    # overhead_pipeline.center_of_mass (weighted average of row/col indices).
    H, W = mask.shape
    total = mask.sum()
    if total <= 0:
        return -1.0, -1.0
    rows = np.arange(H)
    cols = np.arange(W)
    row = (mask * rows[:, None]).sum() / total
    col = (mask * cols[None, :]).sum() / total
    return row.item(), col.item()

# sample_id -> predicted mask (out_h, out_w), populated by load_run_df below. Kept as a
# separate dict rather than a samples_df column so the df doesn't carry large arrays.
mask_pred_by_sample = {}

def load_run_df(run_dir=RUN_DIR):
    if run_dir is None:
        return None
    run_dir = Path(run_dir)

    splits = {'train': run_dir / 'train'}
    if (run_dir / 'eval').exists():
        splits.update({f'eval/{p.name}': p for p in sorted((run_dir / 'eval').iterdir())})

    rows = []
    for split, split_dir in splits.items():
        if not split_dir.exists():
            continue
        for pt_path in sorted(split_dir.glob('*.pt')):
            batch = torch.load(pt_path, map_location='cpu', weights_only=False)
            info = batch['info']
            mask_pred = batch['mask_pred'].float().numpy()  # (B, H, W)
            mask_true = batch['mask_true'].float().numpy()  # (B, H, W)

            n = len(info['sample_id'])
            for i in range(n):
                sid = int(info['sample_id'][i])
                pred_row, pred_col = _mask_com(mask_pred[i])
                true_x, true_y = info['x_com'][i].item(), info['y_com'][i].item()
                com_dist = (float('nan') if true_x < 0 or true_y < 0
                           else float(np.hypot(pred_row - true_x, pred_col - true_y)))
                mask_pred_by_sample[sid] = mask_pred[i]
                rows.append({
                    'sample_id':  sid,
                    'split':      split,
                    'pred_com_x': pred_row,
                    'pred_com_y': pred_col,
                    'com_dist':   com_dist,
                    'mse':        float(np.mean((mask_pred[i] - mask_true[i]) ** 2)),
                })
    return pd.DataFrame(rows)

run_df = load_run_df()
run_df

In [ ]:
# Merge run predictions onto samples_df (left join on sample_id) when a run is loaded.
# A sample can appear at most once across splits (each sample belongs to exactly one
# split), so this is a 1:1 merge; samples without a matching run row keep NaN/None.

if run_df is not None:
    samples_df = samples_df.merge(run_df, on='sample_id', how='left')
samples_df

# 0.1 Plotting Functions

In [ ]:
def _match(series, value):
    # value may be a single scalar or a list/tuple/set of allowed values.
    values = [value] if not isinstance(value, (list, tuple, set)) else list(value)
    return series.isin(values)

def _zero_pad(value, width=6):
    # output_id is stored zero-padded (e.g. '000083'); accept plain ints too.
    to_str = lambda v: f'{v:0{width}d}' if isinstance(v, int) else v
    return [to_str(v) for v in value] if isinstance(value, (list, tuple, set)) else to_str(value)

def _as_int_id(series):
    # Coerce an id-like column (sample_id already an int; output_id a zero-padded string
    # like '000083') to plain ints, for numeric coloring/comparison/display.
    return series.astype(int)

def filter_samples_df(df, sample_id=None, speaker=None, n_objects=None, box=None,
                      output_id=None, object=None, is_empty_box=None, split=None,
                      max_samples=None):
    # Each of sample_id/speaker/n_objects/box/output_id/object/is_empty_box/split
    # accepts either a single value or a list of values; rows matching any of them are
    # kept. split is only present after merging in a run_df (see load_run_df).
    df = df.copy()
    if sample_id is not None:    df = df[_match(df['sample_id'], sample_id)]
    if speaker is not None:      df = df[_match(df['speaker'], speaker)]
    if n_objects is not None:    df = df[_match(df['n_objects'], n_objects)]
    if box is not None:          df = df[_match(df['box'], box)]
    if output_id is not None:    df = df[_match(df['output_id'], _zero_pad(output_id))]
    if object is not None:       df = df[_match(df['object'], object)]
    if is_empty_box is not None: df = df[_match(df['is_empty_box'], is_empty_box)]
    if split is not None:        df = df[_match(df['split'], split)]
    if max_samples is not None:
        df = df.iloc[:max_samples]
    return df

In [ ]:
XY_LABELS = ['x', 'y']

def compute_fft_magnitude(fft_path, laser_index=None, xy_index=None, normalize=True,
                          normalize_mode='std-sample'):
    # Core computation: load a sample's raw complex FFT (1, n_lasers, n_freqs, 2 -- last
    # dim = x/y shift direction) and reduce it to a 1D magnitude spectrum (n_freqs,).
    #
    # laser_index: None -> average magnitude over all lasers; int -> that laser only.
    # xy_index:    None -> average magnitude over x and y; 0 -> x only; 1 -> y only.
    # normalize:   apply the same normalization used when building the dataset (via
    #              post_process.normalize_fft) before reducing.
    #
    # Returns (freqs, magnitude), both 1D arrays of shape (n_freqs,).
    with np.load(fft_path) as data:
        fft, freqs = data['fft'], data['freqs']  # fft: (1, L, F, 2) complex

    mag = np.abs(fft.astype(np.complex128)).astype(np.float32)  # (1, L, F, 2)
    if normalize:
        mag = normalize_fft(mag, normalize_mode=normalize_mode)  # matches dataset preprocessing

    mag = mag[0]  # (L, F, 2)
    mag = mag[laser_index] if laser_index is not None else mag.mean(axis=0)  # (F, 2)
    mag = mag[:, xy_index] if xy_index is not None else mag.mean(axis=-1)    # (F,)
    return freqs, mag

In [ ]:
def plot_fft_magnitude(df, sample_id=None, speaker=None, n_objects=None, box=None,
                      output_id=None, object=None, is_empty_box=None, split=None,
                      laser_index=None, xy_index=None, color_by='sample_id',
                      normalize=True, normalize_mode='std-sample', log_y=False,
                      title=None, max_samples=None):
    # Plot each matching sample's FFT magnitude spectrum (see compute_fft_magnitude) as
    # its own line on one figure. Filter kwargs accept a single value or a list of values
    # (see filter_samples_df).
    #
    # color_by: df column to color lines by (e.g. 'sample_id', 'output_id', 'speaker',
    #           'split') -- lines sharing a value get the same color.
    sdf = filter_samples_df(df, sample_id=sample_id, speaker=speaker, n_objects=n_objects,
                            box=box, output_id=output_id, object=object,
                            is_empty_box=is_empty_box, split=split, max_samples=max_samples)
    if sdf.empty:
        print('No samples match the filter.')
        return

    sdf = sdf.assign(output_id=_as_int_id(sdf['output_id']))

    palette = px.colors.qualitative.Plotly
    color_vals = sdf[color_by].unique()
    color_map = {v: palette[i % len(palette)] for i, v in enumerate(color_vals)}

    fig = go.Figure()
    for _, row in sdf.iterrows():
        sid = int(row['sample_id'])
        freqs, mag = compute_fft_magnitude(row['fft_path'], laser_index, xy_index, normalize, normalize_mode)

        # empty box (n_objects == 0) stores com as the sentinel (-1, -1), not a real position
        com_str = (f"({row['com_x']:.1f}, {row['com_y']:.1f})"
                  if pd.notna(row.get('com_x')) and row.get('n_objects') != 0 else 'n/a')

        extra = ''
        if 'split' in row and pd.notna(row.get('split')):
            mse_str = f"{row['mse']:.4g}" if pd.notna(row.get('mse')) else 'n/a'
            dist_str = f"{row['com_dist']:.2f}" if pd.notna(row.get('com_dist')) else 'n/a'
            extra = f"<br>split={row['split']} mse={mse_str} com_dist={dist_str}"

        legend_label = f"id={sid} out={row.get('output_id')}"
        hover_label = (f"id={sid}<br>"
                      f"out={row.get('output_id')} spk={row.get('speaker')} "
                      f"obj={row.get('object')} n_obj={row.get('n_objects')} com={com_str}{extra}")

        fig.add_trace(go.Scatter(
            x=freqs, y=mag, mode='lines', name=legend_label, opacity=0.7,
            line=dict(color=color_map[row[color_by]]),
            hovertemplate=f'{hover_label}<br>freq=%{{x:.1f}} Hz<br>magnitude=%{{y:.4g}}<extra></extra>',
        ))

    laser_desc = 'mean over lasers' if laser_index is None else f'laser={laser_index}'
    xy_desc = 'mean over x/y' if xy_index is None else XY_LABELS[xy_index]
    fig.update_layout(title=title or f'FFT magnitude ({laser_desc}, {xy_desc}, colored by {color_by}, normalize={normalize})',
                      xaxis_title='freq (Hz)', yaxis_title='magnitude',
                      yaxis_type='log' if log_y else 'linear', height=450)
    fig.show()

# 1. Do the various empty-box recordings differ?

In [ ]:
# default: average magnitude over all lasers and both x/y directions
plot_fft_magnitude(samples_df, speaker=1, n_objects=0, color_by='output_id')

The plot shows us that all three of the empty-box recordings at speaker=1 are really similar. This is a good sign, a sanity check that shows that we are robust to real world noise. Across multiple recordings of the same thing, we get the same results. 

# 2. How does the empty box differ from when we put an object in the box?

In [ ]:
plot_fft_magnitude(samples_df, speaker=1, output_id=[83, 84, 85, 0])

For speaker 1 at freq 620, the blue line (object in box) differs from the empty box.

This shows that there is a meaningful distinct signal as we move an object around the box.

In [ ]:
plot_fft_magnitude(samples_df, speaker=4, output_id=[83, 84, 85, 0])

For speaker 4, we also see differences between the box with an object inside and the empty box at freqs 620 (like before) and freqs 200, 400, 480, etc.

# 3. Which speaker is the loudest?

In [ ]:
def compute_speaker_loudness(df, agg='mean', **filter_kwargs):
    # "Loudness" = magnitude of the raw (non-normalized) FFT, averaged over freqs/lasers/xy
    # per sample -- normalize=False on purpose, since std-sample normalization would erase
    # exactly the per-sample amplitude differences we're trying to compare across speakers.
    sdf = filter_samples_df(df, **filter_kwargs)
    loudness = [compute_fft_magnitude(p, normalize=False)[1].mean() for p in sdf['fft_path']]
    sdf = sdf.assign(loudness=loudness)
    return sdf.groupby('speaker')['loudness'].agg(agg).sort_values(ascending=False)

speaker_loudness = compute_speaker_loudness(samples_df)
speaker_loudness

In [ ]:
px.bar(speaker_loudness, title='Mean FFT magnitude by speaker (non-normalized)',
      labels={'value': 'mean magnitude', 'speaker': 'speaker'}).update_layout(showlegend=False)

* It is clear here that speaker 1, 2, are the loudest. Then speakers 3, 4. Then speakers 7,8. Then speakers 5, 6.
* It makes sense this is grouped in pairs because we have the left, right speakers.
* It is weird that there is such a big difference in speaker volume, from 0.1 to 0.4. This is 4x larger fft magnitude!

In [ ]:
def compute_speaker_spectrum(df, agg='mean', normalize=False, normalize_mode='std-sample',
                             laser_index=None, xy_index=None, **filter_kwargs):
    # Per-speaker average FFT magnitude spectrum: for each sample, compute its magnitude
    # spectrum (compute_fft_magnitude), then average across all samples sharing a speaker,
    # frequency-by-frequency. normalize=False by default for the same reason as
    # compute_speaker_loudness -- comparing raw amplitude across speakers.
    sdf = filter_samples_df(df, **filter_kwargs)
    freqs = None
    spectra = {}  # speaker -> list of per-sample magnitude arrays
    for _, row in sdf.iterrows():
        f, mag = compute_fft_magnitude(row['fft_path'], laser_index, xy_index, normalize, normalize_mode)
        freqs = f if freqs is None else freqs
        spectra.setdefault(row['speaker'], []).append(mag)
    return freqs, {speaker: np.stack(mags).mean(axis=0) if agg == 'mean' else np.stack(mags).std(axis=0)
                  for speaker, mags in spectra.items()}

def plot_speaker_spectrum(df, normalize=False, normalize_mode='std-sample',
                          laser_index=None, xy_index=None, log_y=False, **filter_kwargs):
    freqs, spectra = compute_speaker_spectrum(df, normalize=normalize, normalize_mode=normalize_mode,
                                              laser_index=laser_index, xy_index=xy_index, **filter_kwargs)
    palette = px.colors.qualitative.Plotly
    speakers = sorted(spectra.keys())
    color_map = {s: palette[i % len(palette)] for i, s in enumerate(speakers)}

    fig = go.Figure()
    for speaker in speakers:
        fig.add_trace(go.Scatter(
            x=freqs, y=spectra[speaker], mode='lines', name=f'speaker={speaker}',
            line=dict(color=color_map[speaker]),
            hovertemplate=f'speaker={speaker}<br>freq=%{{x:.1f}} Hz<br>magnitude=%{{y:.4g}}<extra></extra>',
        ))

    laser_desc = 'mean over lasers' if laser_index is None else f'laser={laser_index}'
    xy_desc = 'mean over x/y' if xy_index is None else XY_LABELS[xy_index]
    fig.update_layout(title=f'FFT magnitude by speaker, averaged over samples ({laser_desc}, {xy_desc}, normalize={normalize})',
                      xaxis_title='freq (Hz)', yaxis_title='magnitude',
                      yaxis_type='log' if log_y else 'linear', height=450, width=1000)
    fig.show()

plot_speaker_spectrum(samples_df)

We see that at freqs 170, 390, 500, 600 there are big differences in the fft magnitude from one speaker to another.

# 4. What does it look like when we normalize the freqs?

In [ ]:
plot_speaker_spectrum(samples_df, normalize=True)

We normalize with `std-sample`, where each sample has a std of 1.

* This means all speakers are squashed to be in the same domain, regardless of how loud they are.
* There are these weird spikey artifacts at freq 60 and 100. This is especially prouncend for speakers 7, 8, 5, 6. Why?
    * The israeli electrical grid runs at 50hz so the 100hz is likely the second harmonic of that.
* Each speaker still has it's own unique signature.
    * Look at how they have different peak heights at 400 hz.
    * They also taper off differently from freq 650-1000. Maybe those freqs help the model to distinguish between which speaker is which.
    * They oscillate differently from 200-350 hz too.

# 5 What is the resonant freqs of each speaker?

In [ ]:
from scipy.signal import find_peaks

def find_resonant_freqs(freqs, spectra, top_n=10, prominence=None, height=None, min_freq_distance=None):
    # Resonant freqs = peaks in each speaker's magnitude spectrum (spectra, from
    # compute_speaker_spectrum). Returns {speaker: top_n freqs sorted loudest-first}.
    #
    # min_freq_distance: minimum spacing between peaks, in Hz -- find_peaks' own `distance`
    # arg is in bins, not Hz, and freqs isn't 1 Hz/bin (~0.28 Hz/bin here), so we convert.
    distance = None if min_freq_distance is None else max(1, round(min_freq_distance / (freqs[1] - freqs[0])))

    resonant = {}
    for speaker, mag in spectra.items():
        peak_idxs, _ = find_peaks(mag, prominence=prominence, height=height, distance=distance)
        peak_idxs = peak_idxs[np.argsort(-mag[peak_idxs])][:top_n]  # loudest first
        resonant[speaker] = freqs[peak_idxs]
    return resonant

freqs, spectra = compute_speaker_spectrum(samples_df, normalize=True)
resonant_freqs = find_resonant_freqs(freqs, spectra, top_n=8, min_freq_distance=30)
for speaker, peaks in resonant_freqs.items():
    print(f'speaker={speaker}: {np.round(peaks, 1)}')

We compute 8 resonant freqs for each speaker.

We ensure that each peak is at least 30hz away from every other peak.

We compute this on the *normalized* fft magnitude.

In [ ]:
def _lighten(hex_color, amount=0.5):
    # Blend hex_color toward white by `amount` (0=no change, 1=white).
    r, g, b = px.colors.hex_to_rgb(hex_color)
    return f'rgb({r + (255 - r) * amount:.0f}, {g + (255 - g) * amount:.0f}, {b + (255 - b) * amount:.0f})'

def plot_speaker_spectrum_with_peaks(freqs, spectra, resonant, log_y=False):
    # Same as plot_speaker_spectrum, with each speaker's resonant freqs (resonant, from
    # find_resonant_freqs) marked as circles on top of its line, in a lighter shade of
    # that speaker's color.
    palette = px.colors.qualitative.Plotly
    speakers = sorted(spectra.keys())
    color_map = {s: palette[i % len(palette)] for i, s in enumerate(speakers)}

    fig = go.Figure()
    for speaker in speakers:
        fig.add_trace(go.Scatter(
            x=freqs, y=spectra[speaker], mode='lines', name=f'speaker={speaker}',
            line=dict(color=color_map[speaker]),
            hovertemplate=f'speaker={speaker}<br>freq=%{{x:.1f}} Hz<br>magnitude=%{{y:.4g}}<extra></extra>',
        ))
        peak_freqs = resonant[speaker]
        # peaks may not land exactly on a freqs bin -- look up each one's nearest index.
        peak_idxs = [np.argmin(np.abs(freqs - pf)) for pf in peak_freqs]
        peak_mags = spectra[speaker][peak_idxs]
        fig.add_trace(go.Scatter(
            x=peak_freqs, y=peak_mags, mode='markers', name=f'speaker={speaker} peaks', showlegend=False,
            marker=dict(size=9, color=_lighten(color_map[speaker]), line=dict(width=1, color=color_map[speaker])),
            hovertemplate=f'speaker={speaker} resonant peak<br>freq=%{{x:.1f}} Hz<br>magnitude=%{{y:.4g}}<extra></extra>',
        ))

    fig.update_layout(title='FFT magnitude by speaker with resonant peaks',
                      xaxis_title='freq (Hz)', yaxis_title='magnitude',
                      yaxis_type='log' if log_y else 'linear', height=450, width=1000)
    fig.show()

plot_speaker_spectrum_with_peaks(freqs, spectra, resonant_freqs)

We plot the resonant freqs on top of the avg fft magnitude.

This is a visual sanity check that we set top_n and min_freq_distance correctly.

We can visually inspect that we really cover every peak.

In [ ]:
def plot_resonant_freqs_by_speaker(freqs, spectra, resonant):
    # One row per speaker (y axis), resonant freqs as dots along x (Hz); dot size scales
    # with that peak's magnitude so louder resonances stand out at a glance. Each dot is
    # labeled with its exact freq above it.
    palette = px.colors.qualitative.Plotly
    speakers = sorted(resonant.keys())
    color_map = {s: palette[i % len(palette)] for i, s in enumerate(speakers)}

    fig = go.Figure()
    for speaker in speakers:
        peak_freqs = resonant[speaker]
        peak_idxs = [np.argmin(np.abs(freqs - pf)) for pf in peak_freqs]
        peak_mags = spectra[speaker][peak_idxs]
        fig.add_trace(go.Scatter(
            x=peak_freqs, y=[speaker] * len(peak_freqs), mode='markers+text', name=f'speaker={speaker}',
            text=[f'{f:.0f}' for f in peak_freqs], textposition='top center',
            marker=dict(size=8 + 24 * (peak_mags - peak_mags.min()) / (np.ptp(peak_mags) or 1),
                       color=color_map[speaker], line=dict(width=1, color='white')),
            hovertemplate=f'speaker={speaker}<br>freq=%{{x:.1f}} Hz<br>magnitude=%{{customdata:.4g}}<extra></extra>',
            customdata=peak_mags,
        ))

    fig.update_layout(title='Resonant freqs by speaker (dot size = magnitude)',
                      xaxis_title='freq (Hz)', yaxis_title='speaker',
                      yaxis=dict(tickmode='array', tickvals=speakers), height=450, width=1000)
    fig.show()

plot_resonant_freqs_by_speaker(freqs, spectra, resonant_freqs)

* The main resonant freqs are at 171, 388, 419, 480 hz. All speakers have this. 
* We have dummy peaks at 60, 100 hz for speakers 5, 6, 7, 8. Why not for speakers 1, 2, 3, 4?
* The last two resonant freqs change with the speaker and looks like a diagonal line. As we increase the speaker, the resonant freq gets lower.

# 6. What positions are covered?

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from PIL import Image

def plot_com_explorer(df, sample_id=None, speaker=None, n_objects=None, box=None,
                      output_id=None, object=None, is_empty_box=None, split=None,
                      color_by='speaker', max_samples=None):
    # Two-panel COM explorer: left is a scatter of (com_y, com_x) for every matching
    # sample (colored by color_by); hover over a point to load that sample's
    # overhead_path into the image panel on the right. Filter kwargs match filter_samples_df.
    #
    # com_x/com_y come from downsampled_com = (row, col) (see overhead_pipeline.center_of_mass),
    # i.e. com_x is actually the vertical position and com_y the horizontal one -- plot
    # com_y on the x-axis and com_x on the y-axis so the scatter matches the overhead image.
    # Row also increases downward (image coords), so the y-axis is reversed to match. Axis
    # ranges are fixed to the full out_w x out_h grid (not just the filtered points) so the
    # plot doesn't auto-zoom differently depending on what's filtered.
    #
    # color_by: 'sample_id' or 'output_id' get a continuous color scale (brighter/darker
    # by value, via _as_int_id); any other column (e.g. 'speaker') gets discrete
    # qualitative colors, one per value. Hover always shows sample_id and output_id
    # (as plain ints) regardless of color_by. Each point is also labeled with its
    # output_id directly on the plot, above the marker.
    sdf = filter_samples_df(df, sample_id=sample_id, speaker=speaker, n_objects=n_objects,
                            box=box, output_id=output_id, object=object,
                            is_empty_box=is_empty_box, split=split, max_samples=max_samples)
    sdf = sdf[(sdf['n_objects'] != 0)]  # empty-box com is a (-1,-1) sentinel, not a real position
    if sdf.empty:
        print('No samples match the filter.')
        return

    fig = go.FigureWidget()
    hover_ids = lambda d: np.stack([_as_int_id(d['sample_id']), _as_int_id(d['output_id'])], axis=-1)
    labels = lambda d: [str(v) for v in _as_int_id(d['output_id'])]

    if color_by in ('sample_id', 'output_id'):
        fig.add_trace(go.Scatter(
            x=sdf['com_y'], y=sdf['com_x'], mode='markers+text',
            text=labels(sdf), textposition='top center',
            marker=dict(color=_as_int_id(sdf[color_by]), colorscale='Viridis', size=8,
                       colorbar=dict(title=color_by)),
            customdata=hover_ids(sdf),
            hovertemplate='sample=%{customdata[0]}<br>output_id=%{customdata[1]}'
                          '<br>com=(%{y:.1f}, %{x:.1f})<extra></extra>',
        ))
    else:
        palette = px.colors.qualitative.Plotly
        color_vals = sdf[color_by].unique()
        color_map = {v: palette[i % len(palette)] for i, v in enumerate(color_vals)}
        for val in color_vals:
            vdf = sdf[sdf[color_by] == val]
            fig.add_trace(go.Scatter(
                x=vdf['com_y'], y=vdf['com_x'], mode='markers+text', name=f'{color_by}={val}',
                text=labels(vdf), textposition='top center',
                marker=dict(color=color_map[val], size=8),
                customdata=hover_ids(vdf),
                hovertemplate=f'{color_by}={val}<br>sample=%{{customdata[0]}}<br>output_id=%{{customdata[1]}}'
                              '<br>com=(%{y:.1f}, %{x:.1f})<extra></extra>',
            ))

    out_h, out_w = sdf['out_h'].iloc[0], sdf['out_w'].iloc[0]
    fig.update_layout(title='Center of mass (hover a point to see its overhead image)',
                      xaxis_title='horizontal position', yaxis_title='vertical position',
                      xaxis=dict(range=[0, out_w]), yaxis=dict(range=[out_h, 0]),
                      height=450, width=500)

    overhead_by_sample = dict(zip(sdf['sample_id'], sdf['overhead_path']))

    # size the image widget to overhead.png's real aspect ratio so it doesn't get
    # stretched into a square by ipywidgets' default width-only sizing.
    img_w, img_h = Image.open(overhead_by_sample[sdf['sample_id'].iloc[0]]).size
    display_width = 400
    image = widgets.Image(format='png', width=display_width, height=display_width * img_h / img_w)

    def on_hover(trace, points, state):
        if not points.point_inds:
            return
        sid = trace.customdata[points.point_inds[0]][0]
        image.value = Path(overhead_by_sample[sid]).read_bytes()

    for trace in fig.data:
        trace.on_hover(on_hover)

    # show the first sample by default so the image panel isn't empty
    first_sid = sdf['sample_id'].iloc[0]
    image.value = Path(overhead_by_sample[first_sid]).read_bytes()

    display(widgets.HBox([fig, image]))

plot_com_explorer(samples_df, speaker=1, n_objects=1, color_by='output_id')

* We have a pretty uniform grid covering the box.
* We have three main horizontal "lines" covering the top, middle, and bottom of the box. This is because we only had enough room to move the box in 3 places along the y axis. Any other y-coordinates are due to rotations which slightly shift the COM. 

In [ ]:
plot_com_explorer(samples_df, speaker=1, n_objects=1, sample_id=list(range(288)), color_by='output_id')

For samples [0, 287] we put all of the objects uniformly in a grid.

Here we can really see the three horizontal lines.

In [ ]:
plot_com_explorer(samples_df, speaker=1, n_objects=1, sample_id=list(range(288, 568)), color_by='output_id')

For samples [288, 567], we placed all of the objects randomly in the box.

* The following samples move to the right along the bottom: [0, 3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 33]
* The following samples move down:
    * [208, 304, 440, 200, 312, 480, 560, 216] - note: the box is really narrow. So moving down often comes from rotating the object, slightly shifting the COM downward, but without the object actually moving that much vertically.
    * [2, 1, 0]
    * [160, 152, 144]
* The following samples rotate clockwise, starting at 12:
    * [384, 528] - these points have the two closest COM's to each other and so this is really hard
    * [440, 320, 448] 
    * [408, 488, 376]
    * [440, 512, 304, 320, 448]
    * [408, 400, 336, 496, 376, 488]

# 7 How does moving an object horizontally change the fft? 

In [ ]:
import base64

def _b64_image(path):
    return 'data:image/png;base64,' + base64.b64encode(Path(path).read_bytes()).decode()

def plot_com_fft_explorer(df, sample_id=None, speaker=None, n_objects=None, box=None,
                          output_id=None, object=None, is_empty_box=None, split=None,
                          order_by='sample_id', normalize=True, normalize_mode='std-sample',
                          laser_index=None, xy_index=None, max_samples=None, animate=False):
    # Animated three-panel explorer: top row is the COM scatter (each point labeled with
    # its output_id, left) and the current sample's overhead image (right); bottom row
    # (full width, taller than the top row) is every matching sample's FFT magnitude
    # spectrum, with a legend showing each line's sample_id/output_id/com. COM points and
    # FFT lines are colored by output_id (Viridis, shared color map). A play button +
    # slider step through samples in order_by order, highlighting the current sample's
    # COM point and FFT line and swapping in its overhead image on each frame. The
    # slider's current-value text shows sample_id/output_id/com for the current sample,
    # colored to match that sample's FFT line/COM point; each slider step's own label is
    # also broken across three lines (sample/output/com).
    #
    # animate: False (default) renders a static frame only (first sample highlighted, no
    # play button/slider/frames) -- much more compact. True builds the full animation.
    #
    # Trace layout (fixed across all frames): 0/1 = all COM points / all FFT lines
    # (static background); 2/3 = current-sample COM point / FFT line (updated per frame).
    # The overhead image is a layout.images entry (animated via frame.layout).
    sdf = filter_samples_df(df, sample_id=sample_id, speaker=speaker, n_objects=n_objects,
                            box=box, output_id=output_id, object=object,
                            is_empty_box=is_empty_box, split=split, max_samples=max_samples)
    sdf = sdf[(sdf['n_objects'] != 0)]  # empty-box com is a (-1,-1) sentinel, not a real position
    sdf = sdf.sort_values(order_by)
    if sdf.empty:
        print('No samples match the filter.')
        return

    output_ids = _as_int_id(sdf['output_id']).to_numpy()
    oid_min, oid_max = output_ids.min(), output_ids.max()
    color_of = lambda oid: px.colors.sample_colorscale('Viridis', [(oid - oid_min) / ((oid_max - oid_min) or 1)])[0]
    colors = [color_of(oid) for oid in output_ids]
    oid_labels = [str(oid) for oid in output_ids]

    freqs, mags = None, []
    for _, row in sdf.iterrows():
        freqs, mag = compute_fft_magnitude(row['fft_path'], laser_index, xy_index, normalize, normalize_mode)
        mags.append(mag)

    n = len(sdf)
    sample_ids = sdf['sample_id'].to_numpy()
    com_x, com_y = sdf['com_x'].to_numpy(), sdf['com_y'].to_numpy()
    out_h, out_w = sdf['out_h'].iloc[0], sdf['out_w'].iloc[0]
    overhead_paths = sdf['overhead_path'].to_numpy()
    img_w, img_h = Image.open(overhead_paths[0]).size

    legend_of = lambda i: f"id={sample_ids[i]} out={output_ids[i]} com=({com_x[i]:.1f}, {com_y[i]:.1f})"
    # each step's label spans 3 lines: sample, output, com (<br> renders as a line break
    # in Plotly's slider/annotation text, unlike a literal \n).
    detail_of = lambda i: f"sample={sample_ids[i]}<br>output={output_ids[i]}<br>com=({com_x[i]:.1f}, {com_y[i]:.1f})"

    def overhead_image(i):
        return dict(source=_b64_image(overhead_paths[i]), xref='paper', yref='paper',
                   x=1.0, y=1.0, sizex=0.45, sizey=0.45 * img_h / img_w * (900 / 500),
                   xanchor='right', yanchor='top', layer='above')

    fig = make_subplots(rows=2, cols=2, specs=[[{}, {}], [{'colspan': 2}, None]],
                        subplot_titles=('Center of mass', 'Overhead image', f'FFT magnitude (normalize={normalize})'),
                        row_heights=[0.35, 0.65])
    fig.add_trace(go.Scatter(x=com_y, y=com_x, mode='markers+text', text=oid_labels, textposition='top center',
                             marker=dict(color=colors, size=8), showlegend=False), row=1, col=1)
    for j in range(n):
        fig.add_trace(go.Scatter(x=freqs, y=mags[j], mode='lines', line=dict(color=colors[j], width=1),
                                 opacity=0.4, name=legend_of(j), showlegend=True), row=2, col=1)
    fig.add_trace(go.Scatter(x=[com_y[0]], y=[com_x[0]], mode='markers',
                             marker=dict(color=colors[0], size=14, line=dict(color='black', width=1)),
                             showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(x=freqs, y=mags[0], mode='lines', line=dict(color=colors[0], width=3),
                             showlegend=False), row=2, col=1)
    highlight_com_idx, highlight_fft_idx = 1 + n, 2 + n  # trace indices of the two "current" traces
    fig.layout.images = [overhead_image(0)]

    if animate:
        # each frame also restyles the slider's currentvalue text/color to match the
        # current sample's line color, so the sample/output/com readout is colored like
        # its FFT line.
        fig.frames = [
            go.Frame(name=str(i), traces=[highlight_com_idx, highlight_fft_idx], data=[
                go.Scatter(x=[com_y[i]], y=[com_x[i]], mode='markers',
                          marker=dict(color=colors[i], size=14, line=dict(color='black', width=1))),
                go.Scatter(x=freqs, y=mags[i], mode='lines', line=dict(color=colors[i], width=3)),
            ], layout=go.Layout(images=[overhead_image(i)],
                               sliders=[dict(currentvalue=dict(prefix='', suffix='', font=dict(color=colors[i], size=14),
                                                              xanchor='left'))]))
            for i in range(n)
        ]

    title = f'COM + FFT explorer, ordered by {order_by}'
    if speaker is not None:
        title = f'speaker={speaker} -- {title}'
    fig.update_layout(
        title=title, height=900, width=950,
        xaxis=dict(range=[0, out_w], title='horizontal position'),
        yaxis=dict(range=[out_h, 0], title='vertical position'),
        xaxis3=dict(title='freq (Hz)'), yaxis3=dict(title='magnitude'),
        legend=dict(x=1.02, y=0.5, xanchor='left', yanchor='middle'),
    )
    if animate:
        fig.update_layout(
            updatemenus=[dict(type='buttons', showactive=False, x=0, y=-0.12, xanchor='left',
                              buttons=[dict(label='Play', method='animate',
                                           args=[None, dict(frame=dict(duration=400, redraw=True), fromcurrent=True)]),
                                      dict(label='Pause', method='animate',
                                           args=[[None], dict(mode='immediate', frame=dict(duration=0))])])],
            sliders=[dict(active=0, x=0.1, y=-0.12, len=0.85, ticklen=0, pad=dict(t=30),
                         currentvalue=dict(prefix='', suffix='', font=dict(color=colors[0], size=14), xanchor='left'),
                         steps=[dict(label=detail_of(i), method='animate',
                                    args=[[str(i)], dict(mode='immediate', frame=dict(duration=0, redraw=True))])
                               for i in range(n)])],
        )
    fig.show()

speaker = 1
horiztonal_output_ids = [0, 3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 33]
horiztonal_sample_ids = [8*output_id + (speaker-1) for output_id in horiztonal_output_ids]
plot_com_fft_explorer(samples_df, speaker=1, n_objects=1, sample_id=horiztonal_sample_ids, normalize=True)

* For speaker 1, as we move horiztonally each position does have a slightly different FFT. But not in a clear, patter-ed way. Hopefully the pattern will be clear to LLMs though.
* Ideally, the leftmost position would always have lower magnitude or maybe the middle positions have higher peaks, but no such obvious patterns emerge.
* The lines crisscross each other, making this pattern quite complex.
* Remember speaker 1 is on thr right side of the box. So I thought that as we move torwards the right, closer to the speaker, the fft magnitude would increase, i.e. we get louder. But no such thing.

In [ ]:
# speaker = 1
# row = 1
# horiztonal_output_ids = [0, 3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 33]
# horiztonal_sample_ids = [8*(output_id + row) + (speaker-1) for output_id in horiztonal_output_ids]
# plot_com_fft_explorer(samples_df, speaker=speaker, n_objects=1, sample_id=horiztonal_sample_ids, normalize=True)

If we try looking at the middle row of points instead of the bottom row of points, we still cannot see clear patterns.

In [ ]:
# speaker = 4
# row = 1
# horiztonal_output_ids = [0, 3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 33]
# horiztonal_sample_ids = [8*(output_id + row) + (speaker-1) for output_id in horiztonal_output_ids]
# plot_com_fft_explorer(samples_df, speaker=speaker, n_objects=1, sample_id=horiztonal_sample_ids, normalize=True)

Wait! As we use speaker 4, a speaker from the back and we move horiztonally, then we do get a nice pattern!

The lines criss cross each other a lot less.

In [ ]:
# speaker = 2
# row = 1
# horiztonal_output_ids = [0, 3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 33]
# horiztonal_sample_ids = [8*(output_id + row) + (speaker-1) for output_id in horiztonal_output_ids]
# plot_com_fft_explorer(samples_df, speaker=speaker, n_objects=1, sample_id=horiztonal_sample_ids, normalize=True)

# 8 How does moving an object vertically change the fft? 

In [ ]:
speaker = 2
vertical_output_ids = [35, 34, 33]
vertical_sample_ids = [8*output_id + (speaker-1) for output_id in vertical_output_ids]
plot_com_fft_explorer(samples_df, speaker=speaker, n_objects=1, sample_id=vertical_sample_ids, normalize=True)

* We see that as we move down, the peak at 380 hz decreases in magnitude
* At 165 hz, as we move down the box, the peak moves. But this movement is not monatonic. The top position (yellow) has the lowest peak. The bottom position has the middle peak. And the middle position has the highest peak.
* We cannot just eyeball these patterns. 

Maybe a speaker (speaker 4) located behind the box gets a different signal than a speaker to the right of the box (speaker 2)?

In [ ]:
# speaker = 4
# vertical_output_ids = [35, 34, 33]
# vertical_sample_ids = [8*output_id + (speaker-1) for output_id in vertical_output_ids]
# plot_com_fft_explorer(samples_df, speaker=speaker, n_objects=1, sample_id=vertical_sample_ids, normalize=True)


Looking at this plot, it seems to also be different fft for different speakers. But the pattern and differences looks kind of random.

Maybe the problem is that the positions we looked at are at the edges of the box, not the middle of the box.

In [ ]:
# speaker = 2
# vertical_output_ids = [20, 19, 18]
# vertical_sample_ids = [8*output_id + (speaker-1) for output_id in vertical_output_ids]
# plot_com_fft_explorer(samples_df, speaker=speaker, n_objects=1, sample_id=vertical_sample_ids, normalize=True)

Idk, I don't see a pattern here.

# 9 How does rotating change its fft?

In [ ]:
plot_com_fft_explorer(samples_df, speaker=1, n_objects=1, sample_id=[408, 400, 336, 496, 376, 488], normalize=True)

As expected, not much of a clearly visible pattern.

# 10 What does PCA of fft look like?

In [ ]:
from sklearn.decomposition import PCA

def compute_fft_pca(df, n_components=10, normalize=True, normalize_mode='std-sample',
                    laser_index=None, xy_index=None, **filter_kwargs):
    # PCA over every matching sample's normalized FFT magnitude spectrum (see
    # compute_fft_magnitude): each sample is one row (its full freqs-length magnitude
    # vector), so PCA finds the directions of largest variance across samples' spectra.
    # Returns (sdf, pca, components) where components has shape (n_samples, n_components)
    # -- sdf is the filtered/ordered metadata aligned row-for-row with components.
    sdf = filter_samples_df(df, **filter_kwargs)
    sdf = sdf.reset_index(drop=True)
    if sdf.empty:
        print('No samples match the filter.')
        return sdf, None, None

    freqs = None
    X = []
    for _, row in sdf.iterrows():
        f, mag = compute_fft_magnitude(row['fft_path'], laser_index, xy_index, normalize, normalize_mode)
        freqs = f if freqs is None else freqs
        X.append(mag)
    X = np.stack(X)  # (n_samples, n_freqs)

    pca = PCA(n_components=n_components)
    components = pca.fit_transform(X)
    return sdf, pca, components

pca_sdf, pca, pca_components = compute_fft_pca(samples_df, n_components=10, normalize=True)
print(f'explained variance ratio: {np.round(pca.explained_variance_ratio_, 3)}')
print(f'cumulative: {np.round(np.cumsum(pca.explained_variance_ratio_), 3)}')

In [ ]:
pc_labels = [f'PC{i+1}' for i in range(len(pca.explained_variance_ratio_))]
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=pc_labels, y=pca.explained_variance_ratio_, name='explained variance ratio',
    text=[f'{v:.3f}' for v in pca.explained_variance_ratio_], textposition='outside',
))
fig.add_trace(go.Scatter(
    x=pc_labels, y=cumulative_variance, name='cumulative explained variance',
    mode='lines+markers', line=dict(color='red'), yaxis='y2',
))
fig.update_layout(
    title='PCA explained variance ratio (normalized FFT magnitude)',
    xaxis_title='component', yaxis_title='explained variance ratio',
    yaxis2=dict(title='cumulative explained variance', overlaying='y', side='right', range=[0, 1.05]),
    legend=dict(x=0.7, y=0.1),
)
fig.show()

In [ ]:
def plot_pca_2d(sdf, components, color_by='speaker'):
    # PC1 vs PC2 scatter. color_by in ('sample_id', 'output_id') gets a continuous
    # Viridis scale (via _as_int_id); any other column (e.g. 'speaker') gets discrete colors.
    hover_ids = np.stack([_as_int_id(sdf['sample_id']), _as_int_id(sdf['output_id'])], axis=-1)
    hovertemplate = ('sample=%{customdata[0]}<br>output_id=%{customdata[1]}'
                     f'<br>{color_by}=%{{customdata[2]}}<br>PC1=%{{x:.2f}}<br>PC2=%{{y:.2f}}<extra></extra>')

    fig = go.Figure()
    if color_by in ('sample_id', 'output_id'):
        color_vals = _as_int_id(sdf[color_by])
        fig.add_trace(go.Scatter(
            x=components[:, 0], y=components[:, 1], mode='markers',
            marker=dict(color=color_vals, colorscale='Viridis', size=6, colorbar=dict(title=color_by)),
            customdata=np.concatenate([hover_ids, color_vals.to_numpy()[:, None]], axis=-1),
            hovertemplate=hovertemplate,
        ))
    else:
        palette = px.colors.qualitative.Plotly
        color_vals = sdf[color_by].unique()
        color_map = {v: palette[i % len(palette)] for i, v in enumerate(color_vals)}
        for val in color_vals:
            mask = (sdf[color_by] == val).to_numpy()
            fig.add_trace(go.Scatter(
                x=components[mask, 0], y=components[mask, 1], mode='markers', name=f'{color_by}={val}',
                marker=dict(color=color_map[val], size=6),
                customdata=np.concatenate([hover_ids[mask], sdf[color_by][mask].to_numpy()[:, None]], axis=-1),
                hovertemplate=hovertemplate,
            ))

    fig.update_layout(title=f'PCA of normalized FFT magnitude, colored by {color_by}',
                      xaxis_title='PC1', yaxis_title='PC2', height=550, width=700)
    fig.show()

plot_pca_2d(pca_sdf, pca_components, color_by='speaker')

In [ ]:
import colorsys

def _com_to_hsl(com_x, com_y, out_h, out_w):
    # Map each sample's 2D COM position to an HSL color: hue ~ horizontal position
    # (com_y / out_w), lightness ~ vertical position (com_x / out_h) so top/bottom reads
    # as light/dark. Hue is restricted to [0, 0.8] (not the full 0..1 wheel) so it doesn't
    # wrap back to red -- distinct horizontal positions stay visually distinct end-to-end.
    # Lightness spans a wide [0.2, 0.85] range and saturation is maxed at 1.0 so colors
    # stay vivid and clearly separated instead of washed-out pastels.
    hue = 0.8 * (com_y / (out_w or 1)).clip(0, 1)
    lightness = 0.85 - 0.65 * (com_x / (out_h or 1)).clip(0, 1)  # top=light, bottom=dark
    colors = []
    for h, l in zip(hue, lightness):
        r, g, b = colorsys.hls_to_rgb(h, l, 1.0)
        colors.append(f'rgb({r*255:.0f}, {g*255:.0f}, {b*255:.0f})')
    return colors

def plot_pca_2d_by_com(sdf, components, out_h=None, out_w=None):
    # PC1 vs PC2 scatter, colored by each sample's 2D COM position via _com_to_hsl:
    # hue encodes horizontal COM position, lightness encodes vertical COM position.
    # Lets you see whether PCA clusters correspond to COM position/region, using two
    # perceptually distinct color dimensions (hue and light/dark) for the two COM axes.
    out_h = out_h if out_h is not None else sdf['out_h'].iloc[0]
    out_w = out_w if out_w is not None else sdf['out_w'].iloc[0]
    colors = _com_to_hsl(sdf['com_x'].to_numpy(), sdf['com_y'].to_numpy(), out_h, out_w)
    hover_ids = np.stack([_as_int_id(sdf['sample_id']), _as_int_id(sdf['output_id'])], axis=-1)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=components[:, 0], y=components[:, 1], mode='markers',
        marker=dict(color=colors, size=6, line=dict(color='black', width=0.5)),
        customdata=np.concatenate([hover_ids, sdf[['com_x', 'com_y']].to_numpy()], axis=-1),
        hovertemplate='sample=%{customdata[0]}<br>output_id=%{customdata[1]}'
                      '<br>com=(%{customdata[3]:.1f}, %{customdata[2]:.1f})'
                      '<br>PC1=%{x:.2f}<br>PC2=%{y:.2f}<extra></extra>',
    ))

    fig.update_layout(title='PCA of normalized FFT magnitude, colored by COM (hue=horizontal, lightness=vertical)',
                      xaxis_title='PC1', yaxis_title='PC2', height=550, width=700)
    fig.show()

plot_pca_2d_by_com(pca_sdf, pca_components)

In [ ]:
def plot_pca_3d(sdf, components, color_by='speaker', symbol_by=None):
    # PC1/PC2/PC3 3D scatter. color_by in ('sample_id', 'output_id') gets a continuous
    # Viridis scale (via _as_int_id); any other column (e.g. 'speaker') gets discrete
    # colors. symbol_by (optional) additionally varies marker symbol by that column's
    # value (e.g. symbol_by='speaker' while color_by='sample_id'), so two categorical
    # dimensions can be shown at once.
    hover_ids = np.stack([_as_int_id(sdf['sample_id']), _as_int_id(sdf['output_id'])], axis=-1)
    markers = ['circle', 'square', 'diamond', 'cross', 'x', 'triangle-up', 'triangle-down', 'star']
    symbol_map = None
    if symbol_by is not None:
        symbol_vals = sdf[symbol_by].unique()
        symbol_map = {v: markers[i % len(markers)] for i, v in enumerate(symbol_vals)}

    fig = go.Figure()

    def add_trace(mask, name, color, showlegend):
        symbol = [symbol_map[v] for v in sdf[symbol_by][mask]] if symbol_map is not None else 'circle'
        custom = np.concatenate([hover_ids[mask], sdf[color_by][mask].to_numpy()[:, None]], axis=-1)
        extra = f'<br>{symbol_by}=%{{customdata[3]}}' if symbol_by is not None else ''
        if symbol_by is not None:
            custom = np.concatenate([custom, sdf[symbol_by][mask].to_numpy()[:, None]], axis=-1)
        fig.add_trace(go.Scatter3d(
            x=components[mask, 0], y=components[mask, 1], z=components[mask, 2], mode='markers',
            name=name, showlegend=showlegend,
            marker=dict(color=color, size=4, symbol=symbol,
                       **({'colorscale': 'Viridis', 'colorbar': dict(title=color_by)} if isinstance(color, (list, np.ndarray, pd.Series)) else {})),
            customdata=custom,
            hovertemplate=('sample=%{customdata[0]}<br>output_id=%{customdata[1]}'
                          f'<br>{color_by}=%{{customdata[2]}}{extra}'
                          '<br>PC1=%{x:.2f}<br>PC2=%{y:.2f}<br>PC3=%{z:.2f}<extra></extra>'),
        ))

    if color_by in ('sample_id', 'output_id'):
        mask = np.ones(len(sdf), dtype=bool)
        add_trace(mask, color_by, _as_int_id(sdf[color_by]), showlegend=False)
    else:
        palette = px.colors.qualitative.Plotly
        color_vals = sdf[color_by].unique()
        color_map = {v: palette[i % len(palette)] for i, v in enumerate(color_vals)}
        for val in color_vals:
            mask = (sdf[color_by] == val).to_numpy()
            add_trace(mask, f'{color_by}={val}', color_map[val], showlegend=True)

    title = f'3D PCA of normalized FFT magnitude, colored by {color_by}'
    if symbol_by is not None:
        title += f', marker={symbol_by}'
    fig.update_layout(title=title, height=700, width=850,
                      scene=dict(xaxis_title='PC1', yaxis_title='PC2', zaxis_title='PC3'))
    fig.show()

plot_pca_3d(pca_sdf, pca_components, color_by='speaker')

In [ ]:
plot_pca_3d(pca_sdf, pca_components, color_by='output_id')

In [ ]:
def plot_pca_3d_by_com(sdf, components, out_h=None, out_w=None):
    # PC1/PC2/PC3 3D scatter, colored by each sample's 2D COM position via _com_to_hsl:
    # hue encodes horizontal COM position, lightness encodes vertical COM position.
    # Same coloring convention as plot_pca_2d_by_com, extended to 3D.
    out_h = out_h if out_h is not None else sdf['out_h'].iloc[0]
    out_w = out_w if out_w is not None else sdf['out_w'].iloc[0]
    colors = _com_to_hsl(sdf['com_x'].to_numpy(), sdf['com_y'].to_numpy(), out_h, out_w)
    hover_ids = np.stack([_as_int_id(sdf['sample_id']), _as_int_id(sdf['output_id'])], axis=-1)

    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=components[:, 0], y=components[:, 1], z=components[:, 2], mode='markers',
        marker=dict(color=colors, size=4, line=dict(color='black', width=0.5)),
        customdata=np.concatenate([hover_ids, sdf[['com_x', 'com_y']].to_numpy()], axis=-1),
        hovertemplate='sample=%{customdata[0]}<br>output_id=%{customdata[1]}'
                      '<br>com=(%{customdata[3]:.1f}, %{customdata[2]:.1f})'
                      '<br>PC1=%{x:.2f}<br>PC2=%{y:.2f}<br>PC3=%{z:.2f}<extra></extra>',
    ))

    fig.update_layout(title='3D PCA of normalized FFT magnitude, colored by COM (hue=horizontal, lightness=vertical)',
                      height=700, width=850,
                      scene=dict(xaxis_title='PC1', yaxis_title='PC2', zaxis_title='PC3'))
    fig.show()

plot_pca_3d_by_com(pca_sdf, pca_components)

In [ ]:
subset_output_ids = [6, 7, 8, 29, 28, 27]
subset_speakers = [1, 4, 7]
subset_sdf, subset_pca, subset_components = compute_fft_pca(
    samples_df, n_components=10, normalize=True,
    output_id=subset_output_ids, speaker=subset_speakers, n_objects=1)
plot_pca_3d(subset_sdf, subset_components, color_by='sample_id', symbol_by='speaker')

In [ ]:
def _com_to_rgb(com_x, com_y, out_h, out_w):
    # Map each sample's 2D COM position to an RGB color (like a pcolormesh over the COM
    # grid): red ~ horizontal position (com_y / out_w), green ~ vertical position
    # (com_x / out_h), blue fixed -- so points close together in the COM grid get similar
    # colors, and that same color is reused for the PCA panel to check whether COM
    # neighbors are also PCA neighbors.
    r = (com_y / (out_w or 1) * 255).clip(0, 255)
    g = (com_x / (out_h or 1) * 255).clip(0, 255)
    b = np.full_like(r, 128.0)
    return [f'rgb({rr:.0f}, {gg:.0f}, {bb:.0f})' for rr, gg, bb in zip(r, g, b)]

def plot_com_pca_side_by_side(df, speaker, n_components=10, normalize=True, **filter_kwargs):
    # Two panels sharing one color-per-sample, uniform across both panels: color is
    # derived from each sample's 2D COM position (like a pcolormesh over the COM grid --
    # see _com_to_rgb), not output_id. Left is the COM scatter (per speaker), right is
    # the 3D PCA scatter (PC1/PC2/PC3) of that speaker's normalized FFT magnitude
    # spectra. Same color on both sides for a given sample makes it easy to see whether
    # nearby COM points also cluster together in PCA space.
    sdf, pca, components = compute_fft_pca(df, n_components=n_components, normalize=normalize,
                                           speaker=speaker, n_objects=1, **filter_kwargs)
    if sdf.empty:
        return

    out_h, out_w = sdf['out_h'].iloc[0], sdf['out_w'].iloc[0]
    colors = _com_to_rgb(sdf['com_x'].to_numpy(), sdf['com_y'].to_numpy(), out_h, out_w)
    output_ids = _as_int_id(sdf['output_id']).to_numpy()
    labels = [str(oid) for oid in output_ids]
    hover_ids = np.stack([_as_int_id(sdf['sample_id']), output_ids], axis=-1)

    fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'xy'}, {'type': 'scene'}]],
                        subplot_titles=('Center of mass', '3D PCA (PC1/PC2/PC3)'))
    fig.add_trace(go.Scatter(
        x=sdf['com_y'], y=sdf['com_x'], mode='markers+text', text=labels, textposition='top center',
        marker=dict(color=colors, size=10, line=dict(color='black', width=1)), showlegend=False,
        customdata=hover_ids,
        hovertemplate='sample=%{customdata[0]}<br>output_id=%{customdata[1]}'
                      '<br>com=(%{y:.1f}, %{x:.1f})<extra></extra>',
    ), row=1, col=1)
    fig.add_trace(go.Scatter3d(
        x=components[:, 0], y=components[:, 1], z=components[:, 2], mode='markers',
        marker=dict(color=colors, size=4, line=dict(color='black', width=0.5)), showlegend=False,
        customdata=hover_ids,
        hovertemplate='sample=%{customdata[0]}<br>output_id=%{customdata[1]}'
                      '<br>PC1=%{x:.2f}<br>PC2=%{y:.2f}<br>PC3=%{z:.2f}<extra></extra>',
    ), row=1, col=2)

    fig.update_layout(title=f'speaker={speaker} -- COM and 3D PCA, colored by COM position',
                      height=550, width=1100,
                      xaxis=dict(range=[0, out_w], title='horizontal position'),
                      yaxis=dict(range=[out_h, 0], title='vertical position'),
                      scene=dict(xaxis_title='PC1', yaxis_title='PC2', zaxis_title='PC3'))
    fig.show()

plot_com_pca_side_by_side(samples_df, speaker=1)

In [ ]:
def _pca_to_rgb(components):
    # Min-max normalize PC1/PC2/PC3 independently to [0, 255] and pack as 'rgb(r,g,b)'
    # strings, so a point's color directly encodes its own PCA coordinates (as opposed
    # to coloring by some other column like output_id).
    pc = components[:, :3]
    mins, maxs = pc.min(axis=0), pc.max(axis=0)
    ranges = np.where(maxs > mins, maxs - mins, 1)
    scaled = ((pc - mins) / ranges * 255).clip(0, 255)
    return [f'rgb({r:.0f}, {g:.0f}, {b:.0f})' for r, g, b in scaled]

def plot_com_pca_colors(df, speaker, n_components=10, normalize=True, **filter_kwargs):
    # COM scatter for one speaker, where each point's color is an RGB blend of that
    # sample's own PC1/PC2/PC3 (see _pca_to_rgb) -- so points close in PCA space should
    # look similarly colored, and we can visually check whether nearby COM points also
    # sit close together in PCA space.
    sdf, pca, components = compute_fft_pca(df, n_components=n_components, normalize=normalize,
                                           speaker=speaker, n_objects=1, **filter_kwargs)
    if sdf.empty:
        return

    colors = _pca_to_rgb(components)
    output_ids = _as_int_id(sdf['output_id']).to_numpy()
    labels = [str(oid) for oid in output_ids]
    hover_ids = np.stack([_as_int_id(sdf['sample_id']), output_ids], axis=-1)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=sdf['com_y'], y=sdf['com_x'], mode='markers+text', text=labels, textposition='top center',
        marker=dict(color=colors, size=10, line=dict(color='black', width=1)), showlegend=False,
        customdata=hover_ids,
        hovertemplate='sample=%{customdata[0]}<br>output_id=%{customdata[1]}'
                      '<br>com=(%{y:.1f}, %{x:.1f})<extra></extra>',
    ))

    out_h, out_w = sdf['out_h'].iloc[0], sdf['out_w'].iloc[0]
    fig.update_layout(title=f'speaker={speaker} -- COM colored by PC1/PC2/PC3 (RGB blend)',
                      xaxis_title='horizontal position', yaxis_title='vertical position',
                      xaxis=dict(range=[0, out_w]), yaxis=dict(range=[out_h, 0]),
                      height=550, width=600)
    fig.show()

plot_com_pca_colors(samples_df, speaker=5)

In [ ]:
def plot_com_pca_colors_grid(df, speakers=range(1, 9), nrows=2, ncols=4, n_components=10, normalize=True, **filter_kwargs):
    # Same coloring as plot_com_pca_colors (RGB blend of each sample's own PC1/PC2/PC3),
    # but one subplot per speaker laid out in an nrows x ncols grid, so all speakers can
    # be compared at a glance. PCA is fit independently per speaker (same as calling
    # plot_com_pca_colors speaker-by-speaker), so colors are only comparable within a
    # subplot, not across subplots.
    speakers = list(speakers)
    fig = make_subplots(rows=nrows, cols=ncols,
                        subplot_titles=[f'speaker={s}' for s in speakers],
                        horizontal_spacing=0.02, vertical_spacing=0.05)

    for i, speaker in enumerate(speakers):
        row, col = i // ncols + 1, i % ncols + 1
        sdf, pca, components = compute_fft_pca(df, n_components=n_components, normalize=normalize,
                                               speaker=speaker, n_objects=1, **filter_kwargs)
        if sdf.empty:
            continue

        colors = _pca_to_rgb(components)
        output_ids = _as_int_id(sdf['output_id']).to_numpy()
        labels = [str(oid) for oid in output_ids]
        hover_ids = np.stack([_as_int_id(sdf['sample_id']), output_ids], axis=-1)
        out_h, out_w = sdf['out_h'].iloc[0], sdf['out_w'].iloc[0]

        fig.add_trace(go.Scatter(
            x=sdf['com_y'], y=sdf['com_x'], mode='markers', marker=dict(color=colors, size=6, line=dict(color='black', width=0.5)),
            showlegend=False, customdata=hover_ids,
            hovertemplate=f'speaker={speaker}<br>sample=%{{customdata[0]}}<br>output_id=%{{customdata[1]}}'
                          '<br>com=(%{y:.1f}, %{x:.1f})<extra></extra>',
        ), row=row, col=col)
        fig.update_xaxes(range=[0, out_w], row=row, col=col, showticklabels=False)
        fig.update_yaxes(range=[out_h, 0], row=row, col=col, showticklabels=False)

    fig.update_layout(title='COM colored by PC1/PC2/PC3 (RGB blend), per speaker',
                      height=220 * nrows, width=220 * ncols, showlegend=False,
                      margin=dict(l=20, r=20, t=60, b=20))
    fig.show()

plot_com_pca_colors_grid(samples_df)

# 11 Which speaker performs the best?

In [150]:
def plot_speaker_performance_by_split(df, split=None, metrics=('com_dist', 'mse'), metric_labels=None):
    # One figure with two bar-chart panels (one per metric): x=speaker, bar height=mean,
    # error bar=std, numeric mean printed on top of each bar, and the sample count for
    # that speaker printed below each bar (at y=0). The figure title reports the total
    # number of samples shown. Rows with a NaN metric (e.g. no run prediction, or
    # com_dist for an empty-box sentinel com) are dropped before aggregating.
    #
    # split: a single split name to filter to (e.g. 'train', 'eval/foo'); None (default)
    # includes all samples regardless of split in one combined plot. Call this once per
    # split yourself (e.g. in a loop over sorted(samples_df['split'].unique())) to get
    # one figure per split.
    metric_labels = metric_labels or {m: m for m in metrics}
    sdf = df.dropna(subset=list(metrics))
    if split is not None:
        sdf = sdf[sdf['split'] == split]
    if sdf.empty:
        print(f'No samples match split={split!r} with valid {metrics}.')
        return

    n_total = len(sdf)
    stats = sdf.groupby('speaker')[list(metrics)].agg(['mean', 'std', 'count'])
    speakers = stats.index.tolist()

    fig = make_subplots(rows=1, cols=len(metrics),
                        subplot_titles=[metric_labels[m] for m in metrics])
    for i, metric in enumerate(metrics, start=1):
        means = stats[(metric, 'mean')]
        stds = stats[(metric, 'std')]
        counts = stats[(metric, 'count')]
        fig.add_trace(go.Bar(
            x=speakers, y=means, error_y=dict(type='data', array=stds, visible=True),
            text=[f'{m:.3g}' for m in means], textposition='outside',
            name=metric_labels[metric], showlegend=False,
        ), row=1, col=i)
        for speaker, count in zip(speakers, counts):
            fig.add_annotation(x=speaker, y=0, text=f'n={count}', showarrow=False,
                              yshift=-12, row=1, col=i, font=dict(size=10))
        fig.update_xaxes(title_text='speaker', row=1, col=i)
        fig.update_yaxes(title_text=f'{metric_labels[metric]} (mean +/- std)', row=1, col=i)

    split_desc = 'all splits' if split is None else f'split={split}'
    fig.update_layout(title=f'{split_desc} -- performance by speaker (n={n_total} total samples)',
                      height=450, width=950)
    fig.show()

for split in sorted(samples_df['split'].dropna().unique()):
    plot_speaker_performance_by_split(samples_df, split=split, metrics=('com_dist', 'mse'),
                                      metric_labels={'com_dist': 'COM distance', 'mse': 'mask MSE'})

In [151]:
def plot_metric_distribution(df, split=None, speaker=None, metric='com_dist'):
    # Sorted scatter of metric's distribution: samples matching split/speaker are sorted
    # by metric ascending and plotted as individual points, x=rank (sorted index),
    # y=metric value -- no binning, so every sample is its own dot. split=None matches
    # all splits; speaker=None matches all speakers. Rows with a NaN metric (e.g. no run
    # prediction, or com_dist for an empty-box sentinel com) are dropped first.
    sdf = df.dropna(subset=[metric])
    if split is not None:
        sdf = sdf[sdf['split'] == split]
    if speaker is not None:
        sdf = sdf[_match(sdf['speaker'], speaker)]
    if sdf.empty:
        print(f'No samples match split={split!r}, speaker={speaker!r} with a valid {metric}.')
        return

    sdf = sdf.sort_values(metric).reset_index(drop=True)
    hover_ids = np.stack([_as_int_id(sdf['sample_id']), _as_int_id(sdf['output_id']), sdf['speaker']], axis=-1)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=sdf.index, y=sdf[metric], mode='markers', marker=dict(size=5),
        customdata=hover_ids,
        hovertemplate='rank=%{x}<br>sample=%{customdata[0]}<br>output_id=%{customdata[1]}'
                      f'<br>speaker=%{{customdata[2]}}<br>{metric}=%{{y:.3g}}<extra></extra>',
    ))

    split_desc = 'all splits' if split is None else f'split={split}'
    speaker_desc = 'all speakers' if speaker is None else f'speaker={speaker}'
    mean, std = sdf[metric].mean(), sdf[metric].std()
    title = (f'Sorted {metric} ({split_desc}, {speaker_desc}, n={len(sdf)})'
            f'<br><sup>mean={mean:.3g}, std={std:.3g}</sup>')
    fig.update_layout(title=title, xaxis_title='rank (sorted by metric)', yaxis_title=metric,
                      height=450, width=700)
    fig.show()

plot_metric_distribution(samples_df, split='eval/unseen_pos_speaker', speaker=2, metric='com_dist')

# 12 What do the best and worst samples look like?

In [1]:
from overhead_pipeline import speaker_padding

def _unpadded_overhead_size(img_w, img_h, out_w, out_h):
    # overhead.png = draw_speaker(...)'s padded canvas: a gray border of `pad` px added
    # around the original (unpadded) resized image on every side, where
    # pad = speaker_padding(unpadded_w, unpadded_h) = max(unpadded_w, unpadded_h) // 5 --
    # see overhead_pipeline.draw_speaker. mask_pred/pred_com live in that unpadded
    # image's downsampled (out_h, out_w) grid, so to overlay them on overhead.png we need
    # the unpadded image's real pixel size and offset within the padded canvas.
    #
    # unpadded_w/unpadded_h share out_w/out_h's aspect ratio (mask is just a downsample of
    # the unpadded image), and pad = max(unpadded_w, unpadded_h) // 5 with
    # img_w = unpadded_w + 2*pad, img_h = unpadded_h + 2*pad -- solve directly since the
    # aspect ratio pins unpadded_h = unpadded_w * out_h / out_w.
    ratio = out_h / out_w
    # try scale s.t. unpadded_w = s, unpadded_h = s * ratio, pad = max(s, s*ratio) // 5
    # img_w = s + 2*pad, solve for s (pad is an integer floor division, so iterate a tiny
    # search around the closed-form estimate rather than assuming exact algebra holds).
    max_dim_is_w = ratio <= 1
    if max_dim_is_w:
        # pad = s // 5 (w is the max dim) -> img_w = s + 2*(s // 5) ~= s * 1.4
        s_est = img_w / 1.4
    else:
        # pad = (s*ratio) // 5 (h is the max dim) -> img_h = s*ratio + 2*(s*ratio // 5) ~= s*ratio*1.4
        s_est = (img_h / 1.4) / ratio

    best = None
    for s in range(max(1, int(s_est) - 5), int(s_est) + 6):
        uw, uh = s, round(s * ratio)
        pad = speaker_padding(uw, uh)
        cw, ch = uw + 2 * pad, uh + 2 * pad
        err = abs(cw - img_w) + abs(ch - img_h)
        if best is None or err < best[0]:
            best = (err, uw, uh, pad)
    _, unpadded_w, unpadded_h, pad = best
    return unpadded_w, unpadded_h, pad

def _compose_segmask_overlay(row):
    # Build one sample's overhead image (overhead.png -- padded, with the speaker icon)
    # with its predicted segmentation mask (from mask_pred_by_sample) composited on top
    # as a smooth semi-transparent red overlay, plus that mask upsampled to the full
    # padded canvas size (for hover display of the mask's value at each x/y). The mask
    # lives in the *unpadded* resized image's downsampled (out_h, out_w) grid, so we
    # recover that unpadded region's size/offset within the padded overhead.png (see
    # _unpadded_overhead_size), upsample (bilinear) to real pixel resolution, and paste
    # at (pad, pad).
    #
    # Returns (composed_rgb_array, pred_com_xy_or_None, pad, mask_full_or_None):
    # pred_com_xy/mask_full are None if this sample has no run prediction
    # (mask_pred_by_sample miss); pad is the padded border width (px), always returned so
    # callers can place text/markers in the top padded strip regardless of predictions.
    # mask_full is the raw (unnormalized) mask value resampled onto the full padded
    # canvas (NaN outside the unpadded region) for hover display.
    sid = int(row['sample_id'])
    overhead = np.array(Image.open(row['overhead_path']))
    mask_pred = mask_pred_by_sample.get(sid)
    img_h, img_w = overhead.shape[:2]
    if mask_pred is None:
        return overhead, None, speaker_padding(img_w, img_h), None

    out_h, out_w = mask_pred.shape
    unpadded_w, unpadded_h, pad = _unpadded_overhead_size(img_w, img_h, out_w, out_h)

    mask_norm = (mask_pred - mask_pred.min()) / (np.ptp(mask_pred) or 1)
    mask_img = Image.fromarray((mask_norm * 255).astype(np.uint8))
    mask_img = mask_img.resize((unpadded_w, unpadded_h), resample=Image.BILINEAR)
    mask_arr = np.array(mask_img)

    red_overlay = np.zeros((unpadded_h, unpadded_w, 4), dtype=np.uint8)
    red_overlay[..., 0] = 255
    red_overlay[..., 3] = (mask_arr * 0.6).astype(np.uint8)  # alpha ~ mask value

    composed_pil = Image.fromarray(overhead).convert('RGBA')
    composed_pil.paste(Image.fromarray(red_overlay), (pad, pad), mask=Image.fromarray(red_overlay[..., 3]))
    composed = np.array(composed_pil.convert('RGB'))

    # raw (unnormalized) mask value resampled onto the full padded canvas, NaN outside
    # the unpadded region -- lets hover show the true predicted value at each pixel.
    mask_full = np.full((img_h, img_w), np.nan, dtype=np.float32)
    mask_resized = np.array(Image.fromarray(mask_pred.astype(np.float32))
                            .resize((unpadded_w, unpadded_h), resample=Image.BILINEAR))
    mask_full[pad:pad + unpadded_h, pad:pad + unpadded_w] = mask_resized

    pred_com_xy = (pad + row['pred_com_y'] * unpadded_w / out_w, pad + row['pred_com_x'] * unpadded_h / out_h)
    return composed, pred_com_xy, pad, mask_full

def _pluralize(word, n):
    return word if n == 1 else f'{word}s'

def _segmask_title(row, sep='<br>'):
    # 2-line title for one sample's segmask plot: (1) sample_id, (2) com_dist / mse (the
    # run's error metrics). The rest of the sample's info (split/speaker/com/box/
    # n_objects) is drawn onto the plot itself (see _segmask_overlay_text), not the
    # title, since the padded border around the overhead image has room to spare.
    # sep='<br>' for Plotly titles/subplot_titles; pass sep='\n' for plain-text use.
    com_dist_str = f"{row['com_dist']:.3g}" if pd.notna(row.get('com_dist')) else 'n/a'
    mse_str = f"{row['mse']:.3g}" if pd.notna(row.get('mse')) else 'n/a'
    line1 = f"id={int(row['sample_id'])}"
    line2 = f"com_dist={com_dist_str} mse={mse_str}"
    return sep.join([line1, line2])

def _segmask_overlay_text(row):
    # 2-line text drawn onto the top padded strip of the overhead image itself: (1)
    # "<n> <object>(s) in <box> box" (n_objects pluralized, e.g. '1 cube in metal box' /
    # '2 cubes in metal box'); (2) speaker, com, output_id.
    n_objects = row.get('n_objects')
    obj_desc = f"{n_objects} {_pluralize(str(row.get('object')), n_objects)} in {row.get('box')} box"
    com_str = (f"({row['com_x']:.1f}, {row['com_y']:.1f})"
              if pd.notna(row.get('com_x')) and n_objects != 0 else 'n/a')
    output_id = _as_int_id(pd.Series([row['output_id']])).iloc[0]
    line1 = obj_desc
    line2 = f"speaker={row.get('speaker')} com={com_str} output_id={output_id}"
    return f'{line1}<br>{line2}'

def _add_segmask_traces(fig, row, row_col_kwargs=None):
    # Shared trace-adding logic for one sample's segmask panel: the composed image, an
    # invisible hover layer reporting the predicted mask's value at each x/y (mask_full,
    # from _compose_segmask_overlay), the predicted-COM cross marker, and the
    # split/speaker/com/box/n_objects text annotation in the top padded strip. Used by
    # both plot_predicted_segmask (single figure) and plot_best_worst_samples (grid) --
    # row_col_kwargs is {} for the former, {'row': r, 'col': c} for the latter.
    row_col_kwargs = row_col_kwargs or {}
    composed, pred_com_xy, pad, mask_full = _compose_segmask_overlay(row)
    img_h, img_w = composed.shape[:2]

    fig.add_trace(go.Image(z=composed), **row_col_kwargs)
    if mask_full is not None:
        fig.add_trace(go.Heatmap(
            z=mask_full, opacity=0, showscale=False, hoverongaps=False,
            hovertemplate='x=%{x}<br>y=%{y}<br>predicted mask value=%{z:.3g}<extra></extra>',
        ), **row_col_kwargs)
    if pred_com_xy is not None:
        fig.add_trace(go.Scatter(
            x=[pred_com_xy[0]], y=[pred_com_xy[1]], mode='markers',
            marker=dict(color='rgb(255,150,150)', size=10, symbol='cross-thin', line=dict(width=2, color='rgb(255,150,150)')),
            showlegend=False, hoverinfo='skip',
        ), **row_col_kwargs)

    fig.add_annotation(x=img_w / 2, y=pad * 0.24, text=_segmask_overlay_text(row), showarrow=False,
                       font=dict(color='white', size=10), align='center', **row_col_kwargs)

def plot_predicted_segmask(row, height=450, width=400, title=None):
    # Standalone single-sample plot: row's overhead image with its predicted
    # segmentation mask overlaid, a light-red cross at the predicted COM, hover showing
    # the predicted mask's value at each x/y, and split/speaker/com/box/n_objects text
    # drawn in the image's top padded strip (see _add_segmask_traces). Title defaults to
    # _segmask_title(row) (id, com_dist/mse).
    fig = go.Figure()
    _add_segmask_traces(fig, row)

    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False, autorange='reversed')
    fig.update_layout(title=title or _segmask_title(row), height=height, width=width, showlegend=False)
    fig.show()

def plot_best_worst_samples(df, split=None, metric='com_dist', order='worst', n_rows=None, n_cols=6, **filter_kwargs):
    # n_rows x n_cols grid of the best- or worst-performing samples (by metric, among
    # samples matching split/filter_kwargs), each panel built via _add_segmask_traces
    # (same per-sample overlay/hover/annotation as plot_predicted_segmask). Panel titles
    # show sample_id and the metric's value. order='worst' sorts descending (largest
    # metric first); order='best' sorts ascending (smallest metric first).
    #
    # split is kept as its own kwarg (rather than folded into filter_kwargs) since it's
    # central to this plot (shown in the title) -- pass None to include all splits.
    # filter_kwargs accepts the same keys as filter_samples_df (speaker, n_objects, box,
    # sample_id, output_id, object, is_empty_box, max_samples), each a single value or a
    # list of values.
    #
    # n_rows=None (default) shows every matching sample, using as many rows as needed
    # (ceil(n_samples / n_cols)) instead of capping at a fixed n_rows * n_cols.
    sdf = filter_samples_df(df, split=split, **filter_kwargs).dropna(subset=[metric])
    ascending = (order == 'best')
    sdf = sdf.sort_values(metric, ascending=ascending).reset_index(drop=True)
    if sdf.empty:
        print(f'No samples match split={split!r}, {filter_kwargs} with a valid {metric}.')
        return

    if n_rows is None:
        n_rows = -(-len(sdf) // n_cols)  # ceil division
    else:
        sdf = sdf.head(n_rows * n_cols).reset_index(drop=True)

    # com_dist/mse rendered smaller (<span style="font-size:...">) than the id line so
    # the two-line title fits more compactly.
    fig = make_subplots(rows=n_rows, cols=n_cols,
                        subplot_titles=[f"id={row['sample_id']}<br>"
                                       f"<span style='font-size:12px'>com_dist={row['com_dist']:.3g} mse={row['mse']:.3g}</span>"
                                       for _, row in sdf.iterrows()],
                        horizontal_spacing=0.01 / n_cols, vertical_spacing=0.008 / n_rows)

    for i, row in sdf.iterrows():
        r, c = i // n_cols + 1, i % n_cols + 1
        _add_segmask_traces(fig, row, row_col_kwargs=dict(row=r, col=c))
        fig.update_xaxes(visible=False, row=r, col=c)
        fig.update_yaxes(visible=False, row=r, col=c, autorange='reversed')

    # make_subplots auto-adds one title annotation per panel to fig.layout.annotations
    # (in panel order) -- nudge them down slightly so they sit closer to their image.
    for ann in fig.layout.annotations:
        ann.update(yshift=-6)

    split_desc = 'all splits' if split is None else f'split={split}'
    com_dist_mean, com_dist_std = sdf['com_dist'].mean(), sdf['com_dist'].std()
    mse_mean, mse_std = sdf['mse'].mean(), sdf['mse'].std()
    title = (f'{split_desc} -- {order} {len(sdf)} samples by {metric}'
            f'<br><sup>com_dist={com_dist_mean:.3g}+/-{com_dist_std:.3g}, mse={mse_mean:.3g}+/-{mse_std:.3g}</sup>')
    fig.update_layout(title=title, height=340 * n_rows, width=310 * n_cols, showlegend=False,
                      margin=dict(t=110))
    fig.show()

fig = plot_best_worst_samples(samples_df, split='eval/unseen_pos_speaker', metric='com_dist', order='worst', n_cols=4)

ModuleNotFoundError: No module named 'overhead_pipeline'